# Colab 01 - Build Embeddings / Qdrant Index

Run this notebook on Colab GPU to build the real Qdrant index with BioMedBERT and BioCLIP.

## Required Colab Secrets
- `QDRANT_URL`
- `QDRANT_API_KEY`

## Optional Colab Secrets
- `OPENROUTER_API_KEY` for later generation/evaluation
- `HF_TOKEN` only if you need gated/private HuggingFace models

> This notebook defaults BioMedBERT to the public HuggingFace model `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext` because the previous `large` model id is not public/valid and causes 401 errors.


## 1. Clone / update project

If the runtime is fresh, this clones the repository. If it already exists, it pulls the latest committed changes.


In [ ]:
REPO_URL = "https://github.com/phamdinhhai/project-ks2.git"
PROJECT_DIR = "/content/project-ks2"

import os
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!git pull --ff-only || true
!pwd
!find . -maxdepth 2 -name pyproject.toml
!find src -maxdepth 2 -type d -name medical_rag


## 2. Install dependencies

This installs the local package in editable mode so `python -m medical_rag ...` works from Colab.


In [ ]:
!python -m pip install -U pip
!python -m pip install -e ".[gpu,qdrant,agent,eval]"
!python -m pip install -U requests accelerate bitsandbytes qwen-vl-utils huggingface_hub
!python -c "import medical_rag; print('medical_rag import OK')"


## 3. Configure secrets and model defaults

Do not hardcode real keys in this notebook. Put them in Colab Secrets instead.

BioMedBERT default is set to the public base checkpoint. This avoids the 401 error from the old/non-public `large` checkpoint.


In [ ]:
import os

try:
    from google.colab import userdata
    for name in [
        "OPENROUTER_API_KEY",
        "QDRANT_URL",
        "QDRANT_API_KEY",
        "HF_TOKEN",
    ]:
        value = userdata.get(name)
        if value:
            os.environ[name] = value
except Exception as exc:
    print("Colab userdata unavailable:", exc)

os.environ.setdefault("OPENROUTER_MODEL", "google/gemini-2.5-flash")
os.environ.setdefault("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")

# Public BioMedBERT checkpoint. The old large checkpoint is not valid/public on HF.
os.environ.setdefault(
    "BIOMEDBERT_MODEL",
    "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
)
os.environ.setdefault("BIOMEDBERT_DIM", "768")

print("OPENROUTER_API_KEY set:", bool(os.environ.get("OPENROUTER_API_KEY")))
print("OPENROUTER_MODEL:", os.environ.get("OPENROUTER_MODEL"))
print("QDRANT_URL set:", bool(os.environ.get("QDRANT_URL")))
print("QDRANT_API_KEY set:", bool(os.environ.get("QDRANT_API_KEY")))
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
print("BIOMEDBERT_MODEL:", os.environ.get("BIOMEDBERT_MODEL"))
print("BIOMEDBERT_DIM:", os.environ.get("BIOMEDBERT_DIM"))


## 4. Compatibility patch for older commits

Run this cell even if you already pulled latest. It is idempotent and ensures Colab does not use the old hard-coded `large` BioMedBERT model.


In [ ]:
from pathlib import Path

p = Path("src/medical_rag/models/biomedbert.py")
text = p.read_text(encoding="utf-8")
text = text.replace(
    'import logging\nfrom typing import Any',
    'import logging\nimport os\nfrom typing import Any',
)
text = text.replace(
    'DEFAULT_MODEL_NAME = (\n    "microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract-fulltext"\n)\nEMBEDDING_DIM = 1024',
    'DEFAULT_MODEL_NAME = os.environ.get(\n    "BIOMEDBERT_MODEL",\n    "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",\n)\nEMBEDDING_DIM = int(os.environ.get("BIOMEDBERT_DIM", "768"))',
)
p.write_text(text, encoding="utf-8")

!python -m py_compile src/medical_rag/models/biomedbert.py
!python - <<'PY'
from medical_rag.models.biomedbert import DEFAULT_MODEL_NAME, EMBEDDING_DIM
print('DEFAULT_MODEL_NAME =', DEFAULT_MODEL_NAME)
print('EMBEDDING_DIM =', EMBEDDING_DIM)
PY


## 5. Verify GPU and HuggingFace model id


In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

from huggingface_hub import model_info
model_id = os.environ["BIOMEDBERT_MODEL"]
info = model_info(model_id, token=os.environ.get("HF_TOKEN") or None)
print("BioMedBERT model OK:", info.modelId)


## 6. Verify data availability

If GitHub does not contain the large `data/` folder, mount Google Drive and copy/symlink data before indexing.

Qdrant returning 0 points usually means the Colab runtime has no usable dataset/chunks/images.


In [ ]:
from pathlib import Path

data_dir = Path("data")
print("data exists:", data_dir.exists())
if data_dir.exists():
    files = [p for p in data_dir.rglob("*") if p.is_file()]
    print("data file count:", len(files))
    print("first files:")
    for p in files[:30]:
        print(" -", p)
else:
    print("Missing data/. Mount Drive or copy dataset before indexing.")


### Optional: mount Google Drive if `data/` is missing

Uncomment and adapt the copy path if your dataset is stored in Drive.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !rsync -ah --progress /content/drive/MyDrive/KS_Project_2/data/ /content/project-ks2/data/


## 7. Verify Qdrant Cloud auth before long indexing


In [ ]:
!python -m medical_rag test-qdrant --qdrant-url "$QDRANT_URL" --use-cloud-auth


## 8. Optional encoder smoke test

If this is too slow, skip it and go directly to indexing. If it fails with HuggingFace 401, re-check `BIOMEDBERT_MODEL` printed above.


In [ ]:
!python -m medical_rag test-encoders --no-mock --include-bge


## 9. Build real Qdrant index - small first

Use `--recreate` because switching from BioMedBERT large to base changes text vector dimension from 1024 to 768.


In [ ]:
!python -m medical_rag build-qdrant-index \
  --data-dir data \
  --qdrant-url "$QDRANT_URL" \
  --use-cloud-auth \
  --limit 100 \
  --recreate \
  --no-use-mock-models


## 10. Verify point counts after indexing

Expected result after a successful small run: `text_chunks.points > 0` and/or `image_patches.points > 0`.


In [ ]:
!python -m medical_rag test-qdrant --qdrant-url "$QDRANT_URL" --use-cloud-auth


## 11. Full indexing after small run succeeds

Only run this after the small `--limit 100` run has non-zero Qdrant points. Remove `--limit` for full indexing.


In [ ]:
# !python -m medical_rag build-qdrant-index \
#   --data-dir data \
#   --qdrant-url "$QDRANT_URL" \
#   --use-cloud-auth \
#   --recreate \
#   --no-use-mock-models
